# Infra-FM: STAC Imagery Fetch

Fetches Sentinel-2 + Sentinel-1 tiles for infrastructure assets across multiple regions.

**Regions in this notebook:** australia-oceania, africa, south-america

**Before running:**
1. Upload your deduped parquet files to Google Drive:
   - `infra_fm/pipeline/africa_deduped_assets_substations.parquet`
   - `infra_fm/pipeline/australia-oceania_deduped_assets_substations.parquet`
   - `infra_fm/pipeline/south-america_deduped_assets_substations.parquet`
2. Upload your curation code zip to Drive: `infra_fm/code/infra_fm_curation.zip`
3. Run cells top to bottom — checkpoints save to Drive so you can resume if disconnected.

## 1. Mount Google Drive

In [1]:
import psutil
import subprocess

ram = psutil.virtual_memory()
print(f'RAM: {ram.available/1e9:.1f}GB available / {ram.total/1e9:.1f}GB total')
print(f'Used: {ram.used/1e9:.1f}GB ({ram.percent:.0f}%)')

# Check top memory consumers
result = subprocess.run(['ps', 'aux', '--sort=-%mem'], 
                       capture_output=True, text=True)
lines = result.stdout.split('\n')
print('\nTop memory consumers:')
for line in lines[:10]:
    print(line)

RAM: 12.4GB available / 13.6GB total
Used: 0.9GB (9%)

Top memory consumers:
USER         PID %CPU %MEM    VSZ   RSS TTY      STAT START   TIME COMMAND
root       13183  0.2  1.6 1014140 217576 ?      Ssl  May03   0:08 /usr/bin/python3 -m colab_kernel_launcher -f /root/.local/share/jupyter/runtime/kernel-e9126712-082b-4218-87dd-5aa7b1f82d69.json
root          87  0.1  1.1 414388 148568 ?       Sl   May03   0:10 /usr/bin/python3 /usr/local/bin/jupyter-server --debug --transport="ipc" --ip=172.28.0.12 --ServerApp.token= --port=9000 --FileContentsManager.root_dir=/ --FileContentsManager.allow_hidden=True --ServerApp.log_format="|%(levelname)s|%(message)s" --ServerApp.iopub_data_rate_limit=1e10 --MappingKernelManager.root_dir=/content
root       26027 23.6  0.7 664176 102784 ?       Ssl  00:47   0:01 /usr/bin/python3 -m colab_kernel_launcher -f /root/.local/share/jupyter/runtime/kernel-1d4bcc9f-a787-4f66-ba1a-5bf940a434c5.json
root       21634  0.1  0.7 664032 98936 ?        Ssl  00:29   0

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Verify your files are visible
import os
DRIVE_ROOT = '/content/drive/MyDrive/infra_fm'
print('Drive root contents:')
for f in sorted(os.listdir(DRIVE_ROOT)):
    print(' ', f)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive root contents:
  checkpoints
  code
  datasets
  pipeline
  results
  temp power_only data


## 2. Install dependencies

In [3]:
!pip install -q pystac-client planetary-computer rasterio scipy opencv-python-headless

## 3. Set up curation code

In [4]:
import zipfile, os

ZIP_PATH   = '/content/drive/MyDrive/infra_fm/code/infra_fm_curation.zip'
EXTRACT_TO = '/content'

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    # Find all curation files we need
    needed = [
        'stac_imagery.py', 'qc.py', 'triage.py', 'dataset.py', 'sources.py',
        'helpers/tile_types.py', 'utils/io_utils.py', 
        'utils/timing_log_utils.py', 'legacy/imagery.py'
    ]
    
    for member in z.namelist():
        # Normalize Windows backslashes
        clean = member.replace('\\', '/')
        
        # Check if this is one of the files we need
        for needed_file in needed:
            if clean.endswith(f'curation/{needed_file}'):
                # Extract just this file to /content/
                target = os.path.join(EXTRACT_TO, needed_file)
                os.makedirs(os.path.dirname(target), exist_ok=True)
                with z.open(member) as src, open(target, 'wb') as dst:
                    dst.write(src.read())
                print(f'Extracted: {needed_file}')
                break

# Verify
print('\nVerification:')
print('stac_imagery.py exists:', os.path.exists('/content/stac_imagery.py'))
print('qc.py exists:', os.path.exists('/content/qc.py'))
print('triage.py exists:', os.path.exists('/content/triage.py'))

Extracted: dataset.py
Extracted: qc.py
Extracted: sources.py
Extracted: stac_imagery.py
Extracted: triage.py
Extracted: helpers/tile_types.py
Extracted: legacy/imagery.py
Extracted: utils/io_utils.py
Extracted: utils/timing_log_utils.py

Verification:
stac_imagery.py exists: True
qc.py exists: True
triage.py exists: True


In [5]:
import sys, os
sys.path.insert(0, '/content')
sys.path.insert(0, '/content/utils')
os.chdir('/content')

# Test imports
try:
    from stac_imagery import STACImageryFetcher
    print('stac_imagery OK')
except Exception as e:
    print(f'stac_imagery FAILED: {e}')

try:
    from qc import QualityChecker
    print('qc OK')
except Exception as e:
    print(f'qc FAILED: {e}')

try:
    from triage import RuleBasedTriager
    print('triage OK')
except Exception as e:
    print(f'triage FAILED: {e}')

try:
    from dataset import DatasetAssembler
    print('dataset OK')
except Exception as e:
    print(f'dataset FAILED: {e}')

stac_imagery OK
qc OK
triage OK
dataset OK


In [6]:
import psutil
import subprocess

ram = psutil.virtual_memory()
print(f'RAM: {ram.available/1e9:.1f}GB available / {ram.total/1e9:.1f}GB total')
print(f'Used: {ram.used/1e9:.1f}GB ({ram.percent:.0f}%)')

# Check top memory consumers
result = subprocess.run(['ps', 'aux', '--sort=-%mem'], 
                       capture_output=True, text=True)
lines = result.stdout.split('\n')
print('\nTop memory consumers:')
for line in lines[:10]:
    print(line)

RAM: 12.3GB available / 13.6GB total
Used: 1.0GB (10%)

Top memory consumers:
USER         PID %CPU %MEM    VSZ   RSS TTY      STAT START   TIME COMMAND
root       26027 34.1  1.7 1000572 229656 ?      Ssl  00:47   0:06 /usr/bin/python3 -m colab_kernel_launcher -f /root/.local/share/jupyter/runtime/kernel-1d4bcc9f-a787-4f66-ba1a-5bf940a434c5.json
root       13183  0.2  1.6 1014140 217576 ?      Ssl  May03   0:08 /usr/bin/python3 -m colab_kernel_launcher -f /root/.local/share/jupyter/runtime/kernel-e9126712-082b-4218-87dd-5aa7b1f82d69.json
root          87  0.1  1.1 414388 148568 ?       Sl   May03   0:10 /usr/bin/python3 /usr/local/bin/jupyter-server --debug --transport="ipc" --ip=172.28.0.12 --ServerApp.token= --port=9000 --FileContentsManager.root_dir=/ --FileContentsManager.allow_hidden=True --ServerApp.log_format="|%(levelname)s|%(message)s" --ServerApp.iopub_data_rate_limit=1e10 --MappingKernelManager.root_dir=/content
root       21634  0.1  0.7 664032 98936 ?        Ssl  00:29   

## 4. Configuration — edit this cell before running

In [7]:
# --- Paths ---
PIPELINE_DIR   = f'{DRIVE_ROOT}/pipeline'       # where your deduped parquets live
DATASETS_DIR   = f'{DRIVE_ROOT}/datasets'       # where curated datasets will be written
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints'    # fetch checkpoints — survives disconnects

os.makedirs(DATASETS_DIR,   exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# --- Fetch settings ---
MODALITIES   = ['sentinel2_ms', 'sentinel1']   # landsat_thermal dropped for speed
BUFFER_M     = 300
MAX_WORKERS  = 16    # Colab has good network — push concurrency higher than local
START_WORKERS = 8

# --- Regions to process (in order — smallest first) ---
REGIONS = [
    'europe'
]

# Skip regions that already have a _SUCCESS file
SKIP_DONE = True

print('Configuration:')
print(f'  Modalities:    {MODALITIES}')
print(f'  Buffer:        {BUFFER_M}m')
print(f'  Workers:       {START_WORKERS} start / {MAX_WORKERS} max')
print(f'  Regions:       {REGIONS}')
print(f'  Datasets dir:  {DATASETS_DIR}')
print(f'  Checkpoints:   {CHECKPOINT_DIR}')

Configuration:
  Modalities:    ['sentinel2_ms', 'sentinel1']
  Buffer:        300m
  Workers:       8 start / 16 max
  Regions:       ['europe']
  Datasets dir:  /content/drive/MyDrive/infra_fm/datasets
  Checkpoints:   /content/drive/MyDrive/infra_fm/checkpoints


## Debug

In [8]:
import psutil, os

# Overall RAM
ram = psutil.virtual_memory()
print(f'RAM: {ram.available/1e9:.1f}GB available / {ram.total/1e9:.1f}GB total')
print(f'Used: {ram.used/1e9:.1f}GB ({ram.percent:.0f}%)')

# Check if the infra_fm folder from the old extraction is eating space
import shutil
for folder in ['/content/infra_fm']:
    if os.path.exists(folder):
        size = shutil.disk_usage(folder).used / 1e9
        print(f'{folder}: {size:.2f} GB on disk')

RAM: 12.3GB available / 13.6GB total
Used: 1.0GB (10%)


## 5. Verify deduped parquets are accessible

In [9]:
from pathlib import Path
import pandas as pd

for region in REGIONS:
    path = f'{PIPELINE_DIR}/{region}_deduped_assets_substations_sampled.parquet'
    if Path(path).exists():
        # Read only first row to verify file exists — don't load full dataframe
        df_peek = pd.read_parquet(path, columns=['asset_id'])
        print(f'{region:25s} {len(df_peek):>6,} assets — OK')
        del df_peek  # immediately free memory
    else:
        print(f'{region:25s} MISSING — {path}')

europe                    127,244 assets — OK


In [10]:
import psutil
import subprocess

ram = psutil.virtual_memory()
print(f'RAM: {ram.available/1e9:.1f}GB available / {ram.total/1e9:.1f}GB total')
print(f'Used: {ram.used/1e9:.1f}GB ({ram.percent:.0f}%)')

# Check top memory consumers
result = subprocess.run(['ps', 'aux', '--sort=-%mem'], 
                       capture_output=True, text=True)
lines = result.stdout.split('\n')
print('\nTop memory consumers:')
for line in lines[:10]:
    print(line)

RAM: 12.2GB available / 13.6GB total
Used: 1.1GB (10%)

Top memory consumers:
USER         PID %CPU %MEM    VSZ   RSS TTY      STAT START   TIME COMMAND
root       26027 33.3  2.1 1430104 282984 ?      Ssl  00:47   0:06 /usr/bin/python3 -m colab_kernel_launcher -f /root/.local/share/jupyter/runtime/kernel-1d4bcc9f-a787-4f66-ba1a-5bf940a434c5.json
root       13183  0.2  1.6 1014140 217576 ?      Ssl  May03   0:08 /usr/bin/python3 -m colab_kernel_launcher -f /root/.local/share/jupyter/runtime/kernel-e9126712-082b-4218-87dd-5aa7b1f82d69.json
root          87  0.1  1.1 414388 148568 ?       Sl   May03   0:10 /usr/bin/python3 /usr/local/bin/jupyter-server --debug --transport="ipc" --ip=172.28.0.12 --ServerApp.token= --port=9000 --FileContentsManager.root_dir=/ --FileContentsManager.allow_hidden=True --ServerApp.log_format="|%(levelname)s|%(message)s" --ServerApp.iopub_data_rate_limit=1e10 --MappingKernelManager.root_dir=/content
root       21634  0.1  0.7 664032 98936 ?        Ssl  00:29   

## 6. Run STAC fetch pipeline — all regions

This cell runs the full pipeline (fetch → QC → triage → assemble) for each region.
Checkpoints are saved to Drive every 200 tiles — if the session disconnects, re-run
this cell and it will resume from the last checkpoint automatically.

In [ ]:
# make sure import folders are correct
import sys, os
sys.path.insert(0, '/content')
sys.path.insert(0, '/content/utils')  # ← make sure this is here
os.chdir('/content')

# memory check
import psutil
ram = psutil.virtual_memory()
print(f'RAM available: {ram.available / 1e9:.1f} GB / {ram.total / 1e9:.1f} GB total')

CURATION_DIR = '/content'
sys.path.insert(0, CURATION_DIR)
os.chdir(CURATION_DIR)

import json
from datetime import datetime

from stac_imagery import STACImageryFetcher
from qc import QualityChecker
from triage import RuleBasedTriager
from dataset import DatasetAssembler
from io_utils import load_asset_table


def is_done(region):
    success = Path(f'{DATASETS_DIR}/dataset_{region}_stac_v1/_SUCCESS')
    return success.exists()


def run_region(region):
    print('\n' + '=' * 60)
    print(f'Region: {region}  ({datetime.now().strftime("%H:%M:%S")})')
    print('=' * 60)

    if SKIP_DONE and is_done(region):
        print(f'  Already complete — skipping.')
        return

    if region == 'europe':
        parquet = f'{PIPELINE_DIR}/europe_deduped_assets_substations_sampled.parquet'
    else:
        parquet = f'{PIPELINE_DIR}/{region}_deduped_assets_substations.parquet'
    
    df = load_asset_table(parquet)
    print(f'  Loaded {len(df):,} assets')
    del df  # free memory before fetch

    # reload for fetcher
    df = load_asset_table(parquet)
    
    output_dir      = f'{DATASETS_DIR}/dataset_{region}_stac_v1'
    checkpoint_path = f'{CHECKPOINT_DIR}/{region}_fetch.pkl'
    os.makedirs(output_dir, exist_ok=True)

    print(f'  [1/4] Fetching imagery...')
    try:
        fetcher = STACImageryFetcher(
            buffer_m             = BUFFER_M,
            modalities           = MODALITIES,
            temporal_stack       = False,
            checkpoint_path      = checkpoint_path,
            checkpoint_every     = 200,
            adaptive_concurrency = True,
            start_workers        = 4,
            max_workers          = 16,
        )
        print('  Fetcher created OK')
    except Exception as e:
        print(f'  FETCHER CREATION FAILED: {e}')
        import traceback
        traceback.print_exc()
        return

# =============================================================
    import psutil
    ram = psutil.virtual_memory()
    print(f'RAM before fetch: {ram.available/1e9:.1f}GB available')
# =============================================================

    tiles = fetcher.fetch_all(df)
    n_ok  = sum(1 for t in tiles if t.status == 'ok')
    print(f'  Fetched: {n_ok} ok / {len(tiles) - n_ok} failed')

    # --- Step 2: QC ---
    print(f'  [2/4] Quality control...')
    checker    = QualityChecker(min_valid_ratio=0.80)
    qc_results = checker.check_all(tiles, max_workers=4)
    clean      = checker.filter_ok(qc_results)
    print(f'  QC passed: {len(clean)} / {len(tiles)}')

    # --- Step 3: Triage ---
    print(f'  [3/4] Confidence triage...')
    triager        = RuleBasedTriager(contradiction_threshold=3, low_threshold=4)
    triage_results = triager.triage_all(clean, max_workers=4)
    accepted       = triager.filter_accepted(triage_results)
    print(f'  Accepted: {len(accepted)}')

    # --- Step 4: Assemble ---
    print(f'  [4/4] Assembling dataset -> {output_dir}')
    assembler = DatasetAssembler(output_dir)
    summary   = assembler.assemble(accepted, triage_results)

    # Write _SUCCESS
    success_path = Path(output_dir) / '_SUCCESS'
    with open(success_path, 'w') as f:
        json.dump({
            'completed_at':   datetime.utcnow().isoformat() + 'Z',
            'region':         region,
            'n_dataset_tiles': len(summary),
            'modalities':     MODALITIES,
        }, f, indent=2)

    print(f'  Done. {len(summary)} tiles assembled.')
    return len(summary)


# Run all regions
results = {}
for region in REGIONS:
    try:
        n = run_region(region)
        results[region] = n or 'skipped'
    except Exception as e:
        print(f'ERROR in {region}: {e}')
        results[region] = f'error: {e}'

print('\n' + '=' * 40)
print('SUMMARY')
print('=' * 40)
for region, result in results.items():
    print(f'  {region:25s} {result}')

RAM available: 12.2 GB / 13.6 GB total

Region: europe  (00:47:26)
  Loaded 127,244 assets
  [1/4] Fetching imagery...
STACImageryFetcher: modalities=['sentinel2_ms', 'sentinel1'], n_bands=9, temporal_stack=False, buffer_m=300, workers=4 (adaptive)
  Fetcher created OK
RAM before fetch: 2.5GB available
  STAC fetch: 127244 assets pending (0 already checkpointed)
  [10/127244] (0%) ok=0 fail=10 workers=4
  [20/127244] (0%) ok=0 fail=20 workers=4
  [concurrency] workers=8 | throughput=0.0 tiles/s | fail_rate=100.0% | ↓ back off (high fail rate)
  [30/127244] (0%) ok=0 fail=30 workers=8
  [40/127244] (0%) ok=0 fail=40 workers=8
  [concurrency] workers=8 | throughput=0.0 tiles/s | fail_rate=100.0% | ↓ back off (high fail rate)
  [50/127244] (0%) ok=0 fail=50 workers=8
  [60/127244] (0%) ok=0 fail=60 workers=8
  [concurrency] workers=8 | throughput=0.0 tiles/s | fail_rate=100.0% | ↓ back off (high fail rate)
  [70/127244] (0%) ok=1 fail=69 workers=8
  [80/127244] (0%) ok=1 fail=79 workers=8

## 7. Verify completed datasets

In [ ]:
import json

print('Dataset status:')
for region in REGIONS:
    success_path = Path(f'{DATASETS_DIR}/dataset_{region}_stac_v1/_SUCCESS')
    if success_path.exists():
        meta = json.loads(success_path.read_text())
        print(f'  {region:25s} DONE — {meta["n_dataset_tiles"]:,} tiles')
    else:
        # Check if partially complete via checkpoint
        ckpt = Path(f'{CHECKPOINT_DIR}/{region}_fetch.pkl')
        if ckpt.exists():
            import pickle
            data = pickle.load(open(ckpt, 'rb'))
            n_done = len(data.get('completed_ids', []))
            print(f'  {region:25s} IN PROGRESS — {n_done:,} tiles fetched so far')
        else:
            print(f'  {region:25s} NOT STARTED')

## 8. Download completed datasets to Drive (already done — they write there directly)

Your datasets are written directly to `My Drive/infra_fm/datasets/`.
To use them locally:
1. Download each `dataset_<region>_stac_v1/` folder from Drive to your local `data/curated_datasets/`
2. Or run pretraining directly from Colab (see pretraining notebook)

In [ ]:
import os

print("Contents of /content/ (top level):")
for item in sorted(os.listdir('/content')):
    if not item.startswith('.') and item not in ['drive', 'sample_data']:
        print(f"  {item}")

print("\nContents of /content/legacy/ (if exists):")
if os.path.exists('/content/legacy'):
    for item in sorted(os.listdir('/content/legacy')):
        print(f"  {item}")
else:
    print("  (does not exist)")

print("\nContents of /content/helpers/ (if exists):")
if os.path.exists('/content/helpers'):
    for item in sorted(os.listdir('/content/helpers')):
        print(f"  {item}")
else:
    print("  (does not exist)")

print("\nContents of /content/utils/ (if exists):")
if os.path.exists('/content/utils'):
    for item in sorted(os.listdir('/content/utils')):
        print(f"  {item}")
else:
    print("  (does not exist)")

print("\nKey files present:")
for f in ['stac_imagery.py', 'qc.py', 'triage.py', 'dataset.py', 'sources.py']:
    print(f"  {f}: {os.path.exists(f'/content/{f}')}")

Contents of /content/ (top level):
  __pycache__
  dataset.py
  helpers
  infra_fm
  legacy
  pipeline.py
  qc.py
  sources.py
  stac_imagery.py
  triage.py
  utils

Contents of /content/legacy/ (if exists):
  __pycache__
  imagery.py

Contents of /content/helpers/ (if exists):
  __pycache__
  tile_types.py

Contents of /content/utils/ (if exists):
  io_utils.py
  timing_log_utils.py

Key files present:
  stac_imagery.py: True
  qc.py: True
  triage.py: True
  dataset.py: True
  sources.py: True
